# <div align="center">بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ</div>
### TEB3123 Machine Learning — Standalone Model Inference
**Candidate:** Muhammad Khaidhir bin Mohd Razin | **Student ID:** 22007399  
**Institution:** Universiti Teknologi PETRONAS (UTP) | **Evaluator:** GA Reeyan Afzal  
*Motto:* "رَّبِّ زِدْنِي عِلْمًا" — *My Lord, increase me in knowledge. (20:114)*

---

# Standalone Inference: Titanic Passenger Survival Prediction

This notebook loads the fitted pipeline saved by **Lab 2** () and its metadata card () to evaluate survival probabilities for arbitrary passenger cohorts without retraining.


In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
MODELS = PROJECT_ROOT / "models"

model_path = MODELS / "titanic_survival.joblib"
card_path = MODELS / "titanic_survival_card.json"

model = joblib.load(model_path)
card = json.loads(card_path.read_text(encoding="utf-8"))

print(f"Loaded model: {card['model']}")
print(f"Decision threshold: {card['decision_threshold']}")
print(f"Test ROC AUC: {card['final_test_roc_auc']}")
print(f"Expected features: {card['features']}")


Loaded model: logistic regression pipeline
Decision threshold: 0.5
Test ROC AUC: 0.837
Expected features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


## Test Scenarios: Evaluating Survival Probabilities

Let's test several passenger profiles to observe how socioeconomic class, gender, age, and fare influence survival likelihood under the calibrated decision threshold.

In [2]:
passengers = pd.DataFrame([
    {"Pclass": 2, "Sex": "female", "Age": 30.0, "SibSp": 0, "Parch": 0, "Fare": 20.0, "Embarked": "S"},
    {"Pclass": 1, "Sex": "female", "Age": 28.0, "SibSp": 1, "Parch": 0, "Fare": 80.0, "Embarked": "C"},
    {"Pclass": 3, "Sex": "male",   "Age": 22.0, "SibSp": 0, "Parch": 0, "Fare": 7.25, "Embarked": "S"},
    {"Pclass": 1, "Sex": "male",   "Age": 45.0, "SibSp": 0, "Parch": 0, "Fare": 52.0, "Embarked": "S"},
    {"Pclass": 3, "Sex": "female", "Age": 10.0, "SibSp": 2, "Parch": 1, "Fare": 15.0, "Embarked": "Q"},
])[card["features"]]

# Predict probabilities
probs = model.predict_proba(passengers)[:, 1]
passengers["Survival_Probability"] = probs.round(3)
passengers["Prediction"] = [
    "Survived" if p >= card["decision_threshold"] else "Did Not Survive"
    for p in probs
]

print("Passenger Inference Results:")
print(passengers[["Pclass", "Sex", "Age", "Fare", "Survival_Probability", "Prediction"]].to_string(index=False))


Passenger Inference Results:
 Pclass    Sex  Age  Fare  Survival_Probability      Prediction
      2 female 30.0 20.00                 0.822        Survived
      1 female 28.0 80.00                 0.906        Survived
      3   male 22.0  7.25                 0.236 Did Not Survive
      1   male 45.0 52.00                 0.404 Did Not Survive
      3 female 10.0 15.00                 0.703        Survived
